# 02 — Persistent Session & Product Orientation

Here we identify the L2 GCOV product properly, see which frequencies and polarization/covariance terms it actually contains, and settle on a frequency to work with.

That choice gets saved to the session, so it's asked once — every later module just reads it back.

## Mapping checkpoint

Once a frequency is chosen, it's worth tying it back to the full scene footprint so it's clear what ground it corresponds to.

In [ ]:
from pathlib import Path

from nisar_utils.bootstrap import setup_workshop
WORKSHOP_ROOT = setup_workshop()

from nisar_utils.config import load_config, save_session
from nisar_utils.workflow import build_profile, resolve_terms
from nisar_utils.gcov import open_gcov

cfg = load_config()
NISAR_FILE = Path(cfg["nisar_file"])

if not NISAR_FILE.exists():
    raise FileNotFoundError(
        f"NISAR file not found: {NISAR_FILE}"
    )

# Identify the actual product from the selected HDF5 file.
profile = build_profile(cfg)

print("=" * 70)
print("NISAR GCOV PRODUCT ORIENTATION")
print("=" * 70)

print("File:", NISAR_FILE)
print("SAR family:", profile.sar_family)
print("Band:", profile.band)
print("Level:", profile.product_level)
print("Product:", profile.product_type)
print("Processing type:", profile.processing_type)
print("GCOV root:", profile.gcov_root)
print("EPSG:", profile.epsg)

frequencies = list(profile.frequencies or [])

if not frequencies:
    raise ValueError(
        "No GCOV frequencies were discovered in the selected product."
    )

print("\nAvailable GCOV frequencies:")
for i, frequency in enumerate(frequencies, start=1):
    print(f"  {i}. {frequency}")


In [ ]:
# Select the frequency for this session.
#
# One available frequency -> automatic selection.
# Multiple available frequencies -> explicit user selection.

if len(frequencies) == 1:

    selected_frequency = frequencies[0]

    print(
        f"\nOnly one GCOV frequency is available. "
        f"Automatically selected: {selected_frequency}"
    )

else:

    while True:

        choice = input(
            f"\nSelect GCOV frequency [1-{len(frequencies)}]: "
        ).strip()

        try:
            index = int(choice) - 1

            if 0 <= index < len(frequencies):
                selected_frequency = frequencies[index]
                break

        except ValueError:
            pass

        print(
            f"Invalid selection. Please enter a number "
            f"between 1 and {len(frequencies)}."
        )

print("\nSelected GCOV frequency:", selected_frequency)


In [ ]:
from nisar_utils.gcov import open_gcov,get_grid_coordinates
from nisar_utils.mapping import scene_extent_wgs84,plot_scene_overview,folium_scene_map
grid=f"{profile.gcov_root}/grids/{selected_frequency}"
with open_gcov(NISAR_FILE) as f: _x,_y=get_grid_coordinates(f,grid)
scene_bounds,_=scene_extent_wgs84(_x,_y,profile.epsg)
print("Selected frequency geographic extent:",scene_bounds)
plot_scene_overview(_x,_y,profile.epsg,title=f"NISAR {profile.sar_family} {selected_frequency} — Product Footprint")


In [ ]:
m=folium_scene_map(_x,_y,profile.epsg,title="NISAR Product Footprint + OpenStreetMap")
m


In [ ]:
# Persist the selected frequency.
#
# Downstream modules read this session value through resolve_frequency().
# Product-derived information remains in ProductProfile rather than being
# duplicated into the session.

save_session({
    "nisar_file": str(NISAR_FILE.resolve()),
    "default_frequency": selected_frequency,
})

print("\nSession updated successfully.")
print("NISAR file:", NISAR_FILE)
print("Selected frequency:", selected_frequency)


In [ ]:
# Resolve the covariance terms for the selected frequency.

freq = selected_frequency

terms, diagonal_terms, off_diagonal_terms = resolve_terms(
    profile,
    freq
)

print("\nSelected frequency:", freq)
print("Covariance terms:", terms)
print("Diagonal terms:", diagonal_terms)
print("Off-diagonal terms:", off_diagonal_terms)


In [ ]:
# Verify HDF5 access and the dynamically discovered GCOV root/frequency.

grid = f"{profile.gcov_root}/grids/{freq}"

with open_gcov(NISAR_FILE) as f:

    print("\n" + "=" * 70)
    print("HDF5 / GCOV VERIFICATION")
    print("=" * 70)

    print("HDF5 access: PASS")
    print("Root objects:", list(f.keys()))
    print("GCOV root:", profile.gcov_root)
    print("GCOV root exists:", profile.gcov_root in f)
    print("Selected grid:", grid)
    print("Selected grid exists:", grid in f)

    if profile.gcov_root not in f:
        raise KeyError(
            f"GCOV root not found: {profile.gcov_root}"
        )

    if grid not in f:
        raise KeyError(
            f"Selected GCOV frequency grid not found: {grid}"
        )


In [ ]:
print("\n" + "=" * 70)
print("MODULE 02 COMPLETE")
print("=" * 70)
print("Product identification : PASS")
print("Frequency selection    : PASS")
print("Session persistence    : PASS")
print("GCOV verification      : PASS")
print("\nThe selected frequency is now available to Modules 03–12.")
